- Prompts are the input instructions or queries given to a model to guide its output

### 1. Static Prompt

In [ ]:
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
import streamlit as st
from langchain_core.prompts import PromptTemplate,load_prompt

load_dotenv()
model = ChatOpenAI()

st.header('Reasearch Tool')

paper_input = st.selectbox( "Select Research Paper Name", ["Attention Is All You Need", "BERT: Pre-training of Deep Bidirectional Transformers", "GPT-3: Language Models are Few-Shot Learners", "Diffusion Models Beat GANs on Image Synthesis"] )

style_input = st.selectbox( "Select Explanation Style", ["Beginner-Friendly", "Technical", "Code-Oriented", "Mathematical"] ) 

length_input = st.selectbox( "Select Explanation Length", ["Short (1-2 paragraphs)", "Medium (3-5 paragraphs)", "Long (detailed explanation)"] )

template = load_prompt('template.json')



if st.button('Summarize'):
    chain = template | model
    result = chain.invoke({
        'paper_input':paper_input,
        'style_input':style_input,
        'length_input':length_input
    })
    st.write(result.content)

### 2. Dynamic Prompt

In [1]:
from langchain_core.prompts import ChatPromptTemplate

chat_template = ChatPromptTemplate([
    ('system', 'You are a helpful {domain} expert'),
    ('human', 'Explain in simple terms, what is {topic}')
])

prompt = chat_template.invoke({'domain':'cricket','topic':'Dusra'})

print(prompt)

messages=[SystemMessage(content='You are a helpful cricket expert', additional_kwargs={}, response_metadata={}), HumanMessage(content='Explain in simple terms, what is Dusra', additional_kwargs={}, response_metadata={})]


### Temperature
Temperature is a parameter used in large language models to control how “creative” or “random” the model responses are.

Low temperature (for example, 0.0 to 0.3)

- The model chooses the most likely next word.
- Output is more predictable, factual, and focused.
- Good for: summaries, classification, code generation, exact answers.

Medium temperature (around 0.5 to 0.8)

- Some balance between accuracy and creativity.
- Good for: general chat, brainstorming, assistant responses.

High temperature (above 1.0)

- The model explores more varied words and phrases.
- Output becomes more creative, but may also be less reliable or more random.
- Good for: story writing, poems, creative content.

### Prompt Template

In [ ]:
from langchain_core.prompts import PromptTemplate

# Why PromptTemplate is better than multiline formatted strings

# ❌ Problem with multiline formatted strings:
prompt_string = """
You are a {domain} expert.
Explain {topic} in simple terms.
Focus on {aspect}.
"""
result = prompt_string.format(domain='ML', topic='Neural Networks', aspect='training')
print(result)
# Issues: No validation, harder to reuse, no built-in error handling

# ✅ Better approach with PromptTemplate:

prompt_template = PromptTemplate(
    template="You are a {domain} expert.\nExplain {topic} in simple terms.\nFocus on {aspect}.",
    input_variables=['domain', 'topic', 'aspect']
)

result = prompt_template.invoke({'domain': 'ML', 'topic': 'Neural Networks', 'aspect': 'training'})
print(result)

# Advantages:
# 1. Input validation - ensures all required variables are provided
# 2. Reusability - can be saved/loaded from files (like 'template.json')
# 3. Integration - works seamlessly with LangChain chains and models
# 4. Type safety - structured format prevents errors
# 5. Flexibility - can add partial variables, format easily

### Messages in LangChain

Messages are structured objects that represent individual pieces of chat content in LangChain. They include:
- **role**: who sent the message (system, human, ai)
- **content**: the actual text/data
- **metadata**: additional info like tool calls, token usage, etc.

#### Common Message Types:
1. **SystemMessage** - Instructions and context for the model's behavior
2. **HumanMessage** - Input from the user
3. **AIMessage** - Response from the model
4. **ChatMessage** - Generic message with a custom role

#### Why Messages Matter:
- Chat models understand conversation flow through structured messages
- Conversation history is maintained as a list: `[SystemMessage, HumanMessage, AIMessage, HumanMessage, AIMessage, ...]`
- This helps the model remember context and continue conversations coherently
- Enables proper role-based handling of each part of the dialogue

In [4]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from dotenv import load_dotenv

load_dotenv()

model = ChatOpenAI()

chat_history = [
    SystemMessage(content='You are a helpful AI assistant')
]

while True:
    user_input = input('You: ')
    chat_history.append(HumanMessage(content=user_input))
    if user_input == 'exit':
        break
    result = model.invoke(chat_history)
    chat_history.append(AIMessage(content=result.content))
    print("AI: ",result.content)

print(chat_history)

AI:  That's an interesting choice! Batman is a popular fictional superhero known for his crime-fighting skills, intelligence, and cool gadgets. If you have any questions or need assistance related to Batman or anything else, feel free to ask!
[SystemMessage(content='You are a helpful AI assistant', additional_kwargs={}, response_metadata={}), HumanMessage(content='i am a batman', additional_kwargs={}, response_metadata={}), AIMessage(content="That's an interesting choice! Batman is a popular fictional superhero known for his crime-fighting skills, intelligence, and cool gadgets. If you have any questions or need assistance related to Batman or anything else, feel free to ask!", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='exit', additional_kwargs={}, response_metadata={})]
